# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [1]:
# Load the libraries as required.
%load_ext dotenv
%dotenv
%run "C:\Users\kamoornani\Assignments\production\01_materials\labs\update_path.py"

import os
import pandas as pd

from utils.logger import get_logger
_logs = get_logger(__name__)

In [2]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [3]:
# Create features (X)
target_col = 'area'
X = fires_dt.drop(columns=[target_col])
_logs.info(f'Features X shape: {X.shape}')
X.info()

2026-01-26 16:17:41,922, 397570274.py, 4, INFO, Features X shape: (517, 12)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 12 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
dtypes: float64(7), int64(3), object(2)
memory usage: 48.6+ KB


In [4]:
# Create target (Y)
Y = fires_dt[target_col]
_logs.info(f'Target Y shape: {Y.shape}')
Y.describe()

2026-01-26 16:17:45,739, 1579327898.py, 3, INFO, Target Y shape: (517,)


count     517.000000
mean       12.847292
std        63.655818
min         0.000000
25%         0.000000
50%         0.520000
75%         6.570000
max      1090.840000
Name: area, dtype: float64

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.impute import SimpleImputer

cat_cols = ['month', 'day']
num_cols = [
    'coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi',
    'temp', 'rh', 'wind', 'rain'
]

# preproc1
pipe_num_simple = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('standardizer', StandardScaler())
])

pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', drop='if_binary'))
])

preproc1 = ColumnTransformer(
    transformers=[
        ('numeric_simple', pipe_num_simple, num_cols),
        ('cat_transform', pipe_cat, cat_cols),
    ],
    remainder='drop'
)

_logs.info('Created ColumnTransformer preproc1.')
preproc1

2026-01-26 16:17:51,255, 1621800077.py, 31, INFO, Created ColumnTransformer preproc1.


,transformers,"[('numeric_simple', ...), ('cat_transform', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [6]:
# preproc2
num_cols_transform = ['rain', 'isi', 'dmc']

num_cols_simple = [
    'coord_x', 'coord_y', 'ffmc', 'dc',
    'temp', 'rh', 'wind'
]

pipe_num_yj = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('standardizer', StandardScaler()),
    ('transform', PowerTransformer(method='yeo-johnson'))
])

preproc2 = ColumnTransformer(
    transformers=[
        ('numeric_std', pipe_num_simple, num_cols_simple),
        ('numeric_yj', pipe_num_yj, num_cols_transform),
        ('cat_transform', pipe_cat, cat_cols),
    ],
    remainder='drop'
)

_logs.info('Created ColumnTransformer preproc2.')
preproc2

2026-01-26 16:17:56,285, 714258240.py, 24, INFO, Created ColumnTransformer preproc2.


,transformers,"[('numeric_std', ...), ('numeric_yj', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [7]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Baseline regressor
reg_baseline = Ridge()

# Advanced regressor
reg_advanced = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

# Pipeline A = preproc1 + baseline
pipe_A = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', reg_baseline)
])

_logs.info('Created model pipeline: pipe_A')
pipe_A

2026-01-26 16:18:01,704, 49029838.py, 19, INFO, Created model pipeline: pipe_A


,steps,"[('preprocessing', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric_simple', ...), ('cat_transform', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [8]:
# Pipeline B = preproc2 + baseline
pipe_B = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', reg_baseline)
])

_logs.info('Created model pipeline: pipe_B')
pipe_B

2026-01-26 16:18:05,633, 2911202565.py, 7, INFO, Created model pipeline: pipe_B


,steps,"[('preprocessing', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric_std', ...), ('numeric_yj', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [9]:
# Pipeline C = preproc1 + advanced model
pipe_C = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', reg_advanced)
])

_logs.info('Created model pipeline: pipe_C')
pipe_C

2026-01-26 16:18:08,547, 1566686558.py, 7, INFO, Created model pipeline: pipe_C


,steps,"[('preprocessing', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric_simple', ...), ('cat_transform', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [10]:
# Pipeline D = preproc2 + advanced model
pipe_D = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', reg_advanced)
])

_logs.info('Created model pipeline: pipe_D')
pipe_D

2026-01-26 16:18:11,420, 3079697362.py, 7, INFO, Created model pipeline: pipe_D


,steps,"[('preprocessing', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric_std', ...), ('numeric_yj', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [11]:
from sklearn.model_selection import train_test_split, GridSearchCV

# Split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

# Scoring (GridSearchCV maximizes score -> use negative errors)
scoring = ['neg_mean_squared_error', 'neg_mean_absolute_error', 'r2']

In [12]:
# Parameter grid (4 combinations)
param_grid_A = {
    'regressor__alpha': [0.01, 0.1, 1.0, 10.0],
}

# Grid search
grid_A = GridSearchCV(
    estimator=pipe_A,
    param_grid=param_grid_A,
    scoring=scoring,
    cv=5,
    refit='neg_mean_squared_error',
    n_jobs=-1
)

# Fit
_logs.info('Fitting grid_A (pipe_A)')
grid_A.fit(X_train, Y_train)

# Result
res_A = pd.DataFrame(grid_A.cv_results_).assign(pipeline='A')
_logs.info(f"grid_A best params: {grid_A.best_params_}")
res_A

2026-01-26 16:18:17,958, 2138311683.py, 17, INFO, Fitting grid_A (pipe_A)
2026-01-26 16:18:23,254, 2138311683.py, 22, INFO, grid_A best params: {'regressor__alpha': 10.0}


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__alpha,params,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,...,rank_test_neg_mean_absolute_error,split0_test_r2,split1_test_r2,split2_test_r2,split3_test_r2,split4_test_r2,mean_test_r2,std_test_r2,rank_test_r2,pipeline
0,0.019811,0.002377,0.011295,0.002824,0.01,{'regressor__alpha': 0.01},-1662.848559,-383.058563,-1113.735607,-7270.693843,...,4,-0.019332,-0.770701,-0.546153,-0.009148,-0.231378,-0.315342,0.299684,4,A
1,0.011831,0.003232,0.008910,0.003990,0.10,{'regressor__alpha': 0.1},-1663.731777,-381.100410,-1116.462548,-7269.669692,...,3,-0.019873,-0.761649,-0.549939,-0.009006,-0.225859,-0.313265,0.297777,3,A
2,0.013432,0.001794,0.008888,0.001000,1.00,{'regressor__alpha': 1.0},-1668.277913,-371.217538,-1126.319817,-7266.565690,...,2,-0.022660,-0.715965,-0.563624,-0.008575,-0.199253,-0.302015,0.287924,2,A
3,0.014997,0.001958,0.007733,0.000268,10.00,{'regressor__alpha': 10.0},-1659.075420,-347.793454,-1093.931996,-7243.718534,...,1,-0.017019,-0.607687,-0.518661,-0.005404,-0.134234,-0.256601,0.255894,1,A


In [13]:
# Parameter grid (4 combinations)
param_grid_B = {
    'regressor__alpha': [0.01, 0.1, 1.0, 10.0],
}

# Grid search
grid_B = GridSearchCV(
    estimator=pipe_B,
    param_grid=param_grid_B,
    scoring=scoring,
    cv=5,
    refit='neg_mean_squared_error',
    n_jobs=-1,
)

# Fit
_logs.info('Fitting grid_B (pipe_B)')
grid_B.fit(X_train, Y_train)

# Result
res_B = pd.DataFrame(grid_B.cv_results_).assign(pipeline='B')
_logs.info(f"grid_B best params: {grid_B.best_params_}")
res_B

2026-01-26 16:18:27,717, 1314169206.py, 17, INFO, Fitting grid_B (pipe_B)
2026-01-26 16:18:27,919, 1314169206.py, 22, INFO, grid_B best params: {'regressor__alpha': 10.0}


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__alpha,params,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,split3_test_neg_mean_squared_error,...,rank_test_neg_mean_absolute_error,split0_test_r2,split1_test_r2,split2_test_r2,split3_test_r2,split4_test_r2,mean_test_r2,std_test_r2,rank_test_r2,pipeline
0,0.025849,0.000613,0.015267,0.008761,0.01,{'regressor__alpha': 0.01},-1663.738543,-387.721678,-860.373896,-7257.932449,...,4,-0.019877,-0.792256,-0.194422,-0.007376,-0.216050,-0.245996,0.286365,4,B
1,0.026903,0.002076,0.020605,0.009951,0.10,{'regressor__alpha': 0.1},-1664.553780,-385.655410,-859.187327,-7256.938925,...,3,-0.020377,-0.782705,-0.192775,-0.007239,-0.211161,-0.242851,0.282829,3,B
2,0.031529,0.012922,0.022545,0.011664,1.00,{'regressor__alpha': 1.0},-1668.704129,-374.257162,-850.557179,-7254.698463,...,2,-0.022921,-0.730016,-0.180794,-0.006928,-0.185589,-0.225250,0.263416,2,B
3,0.026507,0.005489,0.007734,0.001516,10.00,{'regressor__alpha': 10.0},-1657.740455,-346.824287,-821.993721,-7237.271284,...,1,-0.016201,-0.603207,-0.141140,-0.004509,-0.118085,-0.176628,0.220008,1,B


In [14]:
# Parameter grid (4 combinations)
param_grid_C = {
    'regressor__n_estimators': [200, 400],
    'regressor__max_depth': [None, 10],
}

# Grid search
grid_C = GridSearchCV(
    estimator=pipe_C,
    param_grid=param_grid_C,
    scoring=scoring,
    cv=5,
    refit='neg_mean_squared_error',
    n_jobs=-1
)

# Fit
_logs.info('Fitting grid_C (pipe_C)')
grid_C.fit(X_train, Y_train)

# Result
res_C = pd.DataFrame(grid_C.cv_results_).assign(pipeline='C')
_logs.info(f"grid_C best params: {grid_C.best_params_}")
res_C

2026-01-26 16:18:31,963, 2518280250.py, 18, INFO, Fitting grid_C (pipe_C)
2026-01-26 16:18:35,686, 2518280250.py, 23, INFO, grid_C best params: {'regressor__max_depth': None, 'regressor__n_estimators': 200}


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__max_depth,param_regressor__n_estimators,params,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,...,rank_test_neg_mean_absolute_error,split0_test_r2,split1_test_r2,split2_test_r2,split3_test_r2,split4_test_r2,mean_test_r2,std_test_r2,rank_test_r2,pipeline
0,0.551319,0.128450,0.366033,0.163644,None,200,"{'regressor__max_depth': None, 'regressor__n_e...",-3085.200125,-1895.086920,-921.389238,...,4,-0.891238,-7.760102,-0.279127,-0.059716,-0.390480,-1.876133,2.954592,1,C
1,1.288649,0.105544,0.345034,0.036774,None,400,"{'regressor__max_depth': None, 'regressor__n_e...",-3177.283143,-2014.049338,-915.937908,...,3,-0.947686,-8.310009,-0.271559,-0.054308,-0.268125,-1.970337,3.184079,2,C
2,0.806275,0.075472,0.215458,0.094088,10,200,"{'regressor__max_depth': 10, 'regressor__n_est...",-3077.913074,-2006.903672,-932.285294,...,2,-0.886771,-8.276978,-0.294253,-0.059375,-0.369176,-1.977311,3.161403,3,C
3,0.954240,0.042723,0.157692,0.097036,10,400,"{'regressor__max_depth': 10, 'regressor__n_est...",-3146.243536,-2019.082719,-928.274109,...,1,-0.928658,-8.333276,-0.288685,-0.054217,-0.292479,-1.979463,3.190183,4,C


In [15]:
# Parameter grid (4 combinations)
param_grid_D = {
    'regressor__min_samples_leaf': [1, 5],
    'regressor__max_features': [0.5, 1.0],
}

# Grid search
grid_D = GridSearchCV(
    estimator=pipe_D,
    param_grid=param_grid_D,
    scoring=scoring,
    cv=5,
    refit='neg_mean_squared_error',
    n_jobs=-1
)

# Fit
_logs.info('Fitting grid_D (pipe_D)')
grid_D.fit(X_train, Y_train)

# Result
res_D = pd.DataFrame(grid_D.cv_results_).assign(pipeline='D')
_logs.info(f"grid_D best params: {grid_D.best_params_}")
res_D

2026-01-26 16:18:39,855, 3699147011.py, 18, INFO, Fitting grid_D (pipe_D)
2026-01-26 16:18:41,135, 3699147011.py, 23, INFO, grid_D best params: {'regressor__max_features': 0.5, 'regressor__min_samples_leaf': 5}


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__max_features,param_regressor__min_samples_leaf,params,split0_test_neg_mean_squared_error,split1_test_neg_mean_squared_error,split2_test_neg_mean_squared_error,...,rank_test_neg_mean_absolute_error,split0_test_r2,split1_test_r2,split2_test_r2,split3_test_r2,split4_test_r2,mean_test_r2,std_test_r2,rank_test_r2,pipeline
0,0.244038,0.063570,0.159298,0.050630,0.5,1,"{'regressor__max_features': 0.5, 'regressor__m...",-2248.597341,-1017.994579,-1176.953823,...,3,-0.378398,-3.705713,-0.633917,-0.023213,-0.237798,-0.995808,1.369399,3,D
1,0.328172,0.067917,0.112956,0.072515,0.5,5,"{'regressor__max_features': 0.5, 'regressor__m...",-1708.280515,-490.651279,-836.486008,...,1,-0.047182,-1.268052,-0.161259,0.006240,-0.124642,-0.318979,0.478126,1,D
2,0.326321,0.061367,0.115193,0.053844,1.0,1,"{'regressor__max_features': 1.0, 'regressor__m...",-2850.723853,-2056.127128,-834.874399,...,4,-0.747504,-8.504515,-0.159022,-0.051535,-0.324440,-1.957403,3.282122,4,D
3,0.278837,0.095970,0.042040,0.014431,1.0,5,"{'regressor__max_features': 1.0, 'regressor__m...",-1827.757385,-579.856200,-855.776343,...,2,-0.120422,-1.680404,-0.188039,-0.009894,-0.259219,-0.451596,0.619874,2,D


# Evaluate

+ Which model has the best performance?

# Export

+ Save the best performing model to a pickle file.

In [16]:
import pickle
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Determine best model by cross-validation performance (refit metric = neg_mean_squared_error)
grids = {
    'A': grid_A,
    'B': grid_B,
    'C': grid_C,
    'D': grid_D
}

best_pipe_name = max(grids, key=lambda k: grids[k].best_score_)
best_grid = grids[best_pipe_name]

_logs.info(f'Best pipeline (by CV neg MSE): {best_pipe_name}')
_logs.info(f'Best CV neg MSE: {best_grid.best_score_}')
_logs.info(f'Best params: {best_grid.best_params_}')

best_model = best_grid.best_estimator_

2026-01-26 16:18:46,100, 1180567335.py, 16, INFO, Best pipeline (by CV neg MSE): B
2026-01-26 16:18:46,101, 1180567335.py, 17, INFO, Best CV neg MSE: -2144.052729439998
2026-01-26 16:18:46,101, 1180567335.py, 18, INFO, Best params: {'regressor__alpha': 10.0}


In [17]:
# Save best-performing model artifact (pickle)
os.makedirs("./models", exist_ok=True)

model_path = f'./models/forest_fires_best_model__pipe_{best_pipe_name}.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)

_logs.info(f'Saved best model to: {model_path}')

best_pipe_name, model_path

2026-01-26 16:18:50,313, 985516179.py, 8, INFO, Saved best model to: ./models/forest_fires_best_model__pipe_B.pkl


('B', './models/forest_fires_best_model__pipe_B.pkl')

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.